# 11 · 深度学习理论与优化技巧

> 前面会"用"，这一节讲"为什么"和"怎么调得更好"。
> 内容：反向传播、梯度问题、初始化、正则化、归一化、残差、优化算法、AMP。

## 11.1 反向传播（Backpropagation）

反向传播 = **链式法则 + 动态规划**。

设网络是复合函数 $L = f_n(f_{n-1}(\dots f_1(x)))$，则：

$$
\frac{\partial L}{\partial \theta_i}
= \frac{\partial L}{\partial z_n}\cdot
  \frac{\partial z_n}{\partial z_{n-1}}\cdots
  \frac{\partial z_{i+1}}{\partial z_i}\cdot
  \frac{\partial z_i}{\partial \theta_i}
$$

**关键洞察**：前面的梯度可以复用（每层只需算"本层输出的梯度 × 本层局部导数"），
这就是动态规划。计算复杂度与前向同阶 $O(N)$。

### 手动验证一个两层网络

In [ ]:
import torch

torch.manual_seed(0)
# 输入、真实值
x = torch.tensor([1.0, 2.0])
y_true = torch.tensor(1.0)

# 参数
W1 = torch.tensor([[0.5, -0.3], [0.2, 0.8]], requires_grad=True)
b1 = torch.tensor([0.1, -0.1], requires_grad=True)
W2 = torch.tensor([[0.7, -0.5]], requires_grad=True)
b2 = torch.tensor([0.2], requires_grad=True)

# 前向： h = relu(x W1^T + b1);  y = h W2^T + b2;  loss = (y-y_true)^2
h_pre = x @ W1.t() + b1
h = torch.relu(h_pre)
y = h @ W2.t() + b2
loss = (y - y_true) ** 2

# 反向
loss.backward()

# 手算对照
# dy/dh = 2(y-y_true) * W2 ; dh/dh_pre = (h_pre>0)
dy = 2 * (y - y_true)
grad_h = dy * W2[0]                       # (2,)
grad_h_pre = grad_h * (h_pre > 0).float() # relu 导数
grad_W1 = torch.outer(grad_h_pre, x)      # (2,2)
grad_b1 = grad_h_pre

print("autograd W1.grad:\n", W1.grad)
print("手算     grad_W1:\n", grad_W1)
print("一致:", torch.allclose(W1.grad, grad_W1, atol=1e-6))
print("b1 一致:", torch.allclose(b1.grad, grad_b1, atol=1e-6))
print("W2 一致:", torch.allclose(W2.grad, dy * h.unsqueeze(0), atol=1e-6))
print("b2 一致:", torch.allclose(b2.grad, dy, atol=1e-6))

## 11.2 梯度问题：消失与爆炸

深层网络中，梯度是很多 Jacobian 矩阵连乘：

$$
\frac{\partial L}{\partial h_1} = \prod_{i=1}^{n} \frac{\partial h_{i+1}}{\partial h_i}
$$

- 每项谱范数 < 1 → 连乘趋近 0 → **梯度消失**（浅层学不动）
- 每项谱范数 > 1 → 连乘爆炸 → **梯度爆炸**（NaN）

### 演示

In [ ]:
import matplotlib.pyplot as plt

def grad_norm_by_depth(depth, scale, activation="sigmoid"):
    torch.manual_seed(0)
    layers = []
    d = 32
    for _ in range(depth):
        lin = torch.nn.Linear(d, d)
        with torch.no_grad():
            lin.weight.mul_(scale)
        layers.append(lin)
    x = torch.randn(8, d, requires_grad=True)
    h = x
    for lin in layers:
        h = torch.sigmoid(lin(h)) if activation == "sigmoid" else torch.relu(lin(h))
    loss = h.mean()
    loss.backward()
    return x.grad.abs().mean().item()

depths = [1, 2, 5, 10, 20, 40]
norms_sig = [grad_norm_by_depth(d, 1.0, "sigmoid") for d in depths]
norms_relu_small = [grad_norm_by_depth(d, 0.5, "relu") for d in depths]
norms_relu_big = [grad_norm_by_depth(d, 1.5, "relu") for d in depths]

plt.figure(figsize=(8, 4))
plt.semilogy(depths, norms_sig, "o-", label="sigmoid, scale=1.0")
plt.semilogy(depths, norms_relu_small, "s-", label="relu, scale=0.5")
plt.semilogy(depths, norms_relu_big, "^-", label="relu, scale=1.5")
plt.xlabel("网络深度"); plt.ylabel("输入梯度范数（对数）")
plt.title("梯度消失/爆炸"); plt.legend(); plt.grid(alpha=0.3, which="both")
plt.show()

## 11.3 参数初始化

初始化的目标：让每层激活值和梯度的方差保持稳定。

| 方法 | 方差 | 适用 |
|---|---|---|
| Xavier/Glorot | $2/(n_{in}+n_{out})$ | tanh / sigmoid |
| Kaiming/He | $2/n_{in}$ | ReLU / LeakyReLU |
| 正交初始化 | — | RNN 隐状态 |
| 小随机 | 0.01 | 浅层（Old） |

**错误初始化**会导致：
- 全 0 → 所有神经元相同，无法打破对称性
- 太大 → 激活饱和 / 梯度爆炸
- 太小 → 信号逐层衰减

In [ ]:
def activation_std(init_fn, depth=20, d=64, activation=torch.relu):
    torch.manual_seed(0)
    x = torch.randn(256, d)
    h = x
    stds = []
    for _ in range(depth):
        lin = torch.nn.Linear(d, d)
        init_fn(lin.weight)
        torch.nn.init.zeros_(lin.bias)
        h = activation(lin(h))
        stds.append(h.std().item())
    return stds

import torch.nn as nn
import numpy as np

inits = {
    "default(小随机)": lambda w: nn.init.normal_(w, 0, 0.01),
    "Xavier": lambda w: nn.init.xavier_uniform_(w),
    "Kaiming": lambda w: nn.init.kaiming_normal_(w, nonlinearity="relu"),
}

plt.figure(figsize=(8, 4))
for name, fn in inits.items():
    s = activation_std(fn)
    plt.plot(s, label=f"{name} (末层 std={s[-1]:.3f})")
plt.xlabel("层"); plt.ylabel("激活标准差")
plt.title("不同初始化下的激活传播（ReLU）")
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# PyTorch 线性层默认初始化就是 Kaiming 均匀分布
lin = nn.Linear(64, 64)
print("默认 weight 分布范围:", lin.weight.min().item(), "~", lin.weight.max().item())
print("理论 kaiming 范围: ±", (6 / 64) ** 0.5)

## 11.4 激活函数的选择

- **ReLU**：默认选择，快；缺点：负区间梯度为 0（死亡 ReLU）。
- **LeakyReLU / PReLU**：负区间有小斜率，缓解死亡。
- **GELU / SiLU**：平滑，Transformer / 现代 CNN 首选。
- **Sigmoid / Tanh**：隐层基本不用（梯度消失），仅用于输出门控。

## 11.5 归一化层

### BatchNorm

对每个通道，在**batch 维度**上做标准化：
$$
\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}},\qquad y = \gamma\hat{x} + \beta
$$

优点：加速收敛、允许更大 lr、轻微正则效果。
缺点：依赖 batch 大小；小 batch / 变长序列不好用。

### LayerNorm / GroupNorm

- **LayerNorm**：对单个样本的所有特征归一化，不依赖 batch → Transformer、RNN。
- **GroupNorm**：通道分组归一化 → 小 batch 的 CV 任务。

### RMSNorm

只用均方根，省去均值计算 → LLaMA 等大模型。

In [ ]:
x = torch.randn(32, 16, 8, 8) * 5 + 3      # (N,C,H,W)

bn = nn.BatchNorm2d(16)
ln = nn.GroupNorm(4, 16)
gn = nn.GroupNorm(8, 16)

bn.train()
for name, layer in [("BatchNorm", bn), ("GroupNorm", ln), ("GroupNorm2", gn)]:
    y = layer(x)
    print(f"{name:12s} mean={y.mean().item():+.4f} std={y.std().item():.4f}")

# BatchNorm 的 running stats（推理用）
print("\nBN running_mean 形状:", bn.running_mean.shape)

In [ ]:
# 小 batch 下 BatchNorm 的不稳定
x_small = torch.randn(2, 16, 8, 8) * 5 + 3
bn = nn.BatchNorm2d(16)
bn.train()
for i in range(3):
    y = bn(x_small)
    print(f"小 batch 第 {i+1} 次: mean={y.mean().item():+.3f} std={y.std().item():.3f}")
print("-> batch 太小，BN 统计量噪声大（这提示我们：小 batch 用 GroupNorm）")

## 11.6 残差连接（ResNet）

$$
y = x + F(x)
$$

反向传播时：$\frac{\partial y}{\partial x} = I + \frac{\partial F}{\partial x}$，
那个 $I$（恒等项）保证梯度**至少能原样传回去**，这是训练上百层网络的关键。

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.net(x))     # 残差

class TinyResNet(nn.Module):
    def __init__(self, depth=20, ch=32):
        super().__init__()
        self.stem = nn.Conv2d(3, ch, 3, padding=1)
        self.blocks = nn.Sequential(*[ResidualBlock(ch) for _ in range(depth)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(ch, 10))

    def forward(self, x):
        return self.head(self.blocks(self.stem(x)))

net = TinyResNet(20)
x = torch.randn(4, 3, 32, 32)
print("20 层 ResNet 输出:", net(x).shape)
print("参数量:", f"{sum(p.numel() for p in net.parameters()):,}")

# 梯度可以传播到很浅的层
loss = net(x).mean()
loss.backward()
print("stem 卷积梯度范数:", net.stem.weight.grad.norm().item())
print("（若没有残差，20 层后这个值会非常小）")

### 为什么残差有效

- 优化景观更平滑（identity 让 loss 更容易下降）
- 梯度高速公路
- 可以看作"多个浅层网络的集成"（ensemble of paths）

## 11.7 正则化

| 方法 | 原理 |
|---|---|
| L2 / weight_decay | 惩罚大权重 |
| L1 | 稀疏化（少用） |
| Dropout | 随机丢神经元，等价模型集成 |
| Drop path / Stochastic depth | 随机丢整个残差分支 |
| 数据增强 | 扩充数据分布 |
| 标签平滑 | 防过度自信 |
| 早停 | 验证集不再提升就停 |
| Mixup / CutMix | 样本混合（SOTA 常用） |
| 权重平均 EMA | 训练末期权重滑动平均 |

In [ ]:
# Mixup 实现
def mixup(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0))
    x_mix = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return x_mix, y_a, y_b, lam

x = torch.randn(8, 3, 32, 32)
y = torch.randint(0, 10, (8,))
xm, ya, yb, lam = mixup(x, y)
print("mixup lambda:", round(lam, 3), " 混合后形状:", xm.shape)

# 带 mixup 的损失
logits = torch.randn(8, 10)
criterion = nn.CrossEntropyLoss()
loss = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb)
print("mixup loss:", loss.item())

In [ ]:
# 权重 EMA（指数滑动平均）
import copy

class EMA:
    def __init__(self, model, decay=0.99):
        self.decay = decay
        self.shadow = copy.deepcopy(model)
        for p in self.shadow.parameters():
            p.requires_grad = False
    def update(self, model):
        for s, m in zip(self.shadow.parameters(), model.parameters()):
            s.data.mul_(self.decay).add_(m.data, alpha=1 - self.decay)

m = nn.Linear(10, 2)
ema = EMA(m)
# 训练中：ema.update(m)；评估时用 ema.shadow
print("EMA shadow 已创建")

## 11.8 优化算法演进

- **SGD**：$\theta \leftarrow \theta - \eta g$
- **Momentum**：累积历史梯度（惯性），抑制震荡
- **Nesterov**：先按动量走一步再看梯度
- **AdaGrad**：按历史梯度平方缩放 lr（稀疏特征）
- **RMSProp**：AdaGrad 的滑动窗口版
- **Adam**：Momentum + RMSProp + 偏差校正
- **AdamW**：解耦权重衰减（正确做法）
- **LAMB / LARS**：超大 batch 训练

**经验**：
- CV 刷点：`SGD + momentum + cosine`，配合长训练
- 通用/Transformer：`AdamW`
- 不收敛先调 lr，再换优化器

In [ ]:
# 直观比较优化器在病态函数上的表现
def rosenbrock(params):
    x, y = params[0], params[1]
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2

def run_optimizer(opt_cls, steps=2000, **kw):
    p = [torch.tensor(-1.5, requires_grad=True), torch.tensor(1.5, requires_grad=True)]
    opt = opt_cls(p, **kw)
    path = []
    for _ in range(steps):
        opt.zero_grad()
        loss = rosenbrock(p)
        loss.backward()
        opt.step()
        path.append([p[0].item(), p[1].item()])
    return np.array(path), loss.item()

path_sgd, l1 = run_optimizer(torch.optim.SGD, lr=1e-3)
path_mom, l2 = run_optimizer(torch.optim.SGD, lr=1e-3, momentum=0.9)
path_adam, l3 = run_optimizer(torch.optim.Adam, lr=0.05)

xx, yy = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-1, 3, 200))
zz = (1 - xx) ** 2 + 100 * (yy - xx ** 2) ** 2
plt.figure(figsize=(8, 6))
plt.contour(xx, yy, np.log1p(zz), levels=30, cmap="Greys")
plt.plot(path_sgd[:, 0], path_sgd[:, 1], label=f"SGD (loss={l1:.2e})", lw=1)
plt.plot(path_mom[:, 0], path_mom[:, 1], label=f"Momentum (loss={l2:.2e})", lw=1)
plt.plot(path_adam[:, 0], path_adam[:, 1], label=f"Adam (loss={l3:.2e})", lw=1)
plt.plot(1, 1, "r*", ms=15, label="最优(1,1)")
plt.legend(); plt.title("优化器在 Rosenbrock 函数上的轨迹")
plt.xlim(-2, 2); plt.ylim(-1, 3); plt.show()

## 11.9 学习率：最重要的超参数

- **太大**：震荡、不收敛、NaN
- **太小**：收敛慢、卡在局部
- **Warmup**：前几百步从很小线性升到目标 lr，稳定训练（Transformer 必备）
- **衰减**：cosine / step / 指数
- **OneCycle**：先升后降，常带来"超收敛"

### LR Range Test（找合适 lr）
从小到大扫 lr，找 loss 下降最快且未爆炸的区间。

In [ ]:
# 简单 LR range test
def lr_range_test(model, loader, lr_min=1e-5, lr_max=1.0, steps=100):
    opt = torch.optim.SGD(model.parameters(), lr=lr_min)
    mult = (lr_max / lr_min) ** (1 / steps)
    lrs, losses = [], []
    it = iter(loader)
    for s in range(steps):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(loader); x, y = next(it)
        opt.zero_grad()
        loss = nn.CrossEntropyLoss()(model(x), y)
        loss.backward(); opt.step()
        opt.param_groups[0]["lr"] *= mult
        lrs.append(opt.param_groups[0]["lr"]); losses.append(loss.item())
    return lrs, losses

m = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
from torch.utils.data import DataLoader, TensorDataset
dl = DataLoader(TensorDataset(torch.randn(1000, 1, 28, 28), torch.randint(0, 10, (1000,))),
                batch_size=64, shuffle=True)
lrs, losses = lr_range_test(m, dl)
smooth = np.convolve(losses, np.ones(5) / 5, mode="valid")

plt.figure(figsize=(8, 4))
plt.semilogx(lrs[:len(smooth)], smooth)
plt.xlabel("learning rate"); plt.ylabel("loss")
plt.title("LR Range Test（选 loss 下降最陡的 lr）")
plt.grid(alpha=0.3, which="both"); plt.show()
print("建议 lr 取 loss 最低点左侧约一个数量级:", f"{lrs[int(np.argmin(smooth))]:.2e}")

## 11.10 过拟合 vs 欠拟合诊断

| 现象 | 训练 loss | 验证 loss | 对策 |
|---|---|---|---|
| 欠拟合 | 高 | 高 | 更大模型、更多 epoch、调 lr、加特征 |
| 过拟合 | 低 | 高且上升 | 正则、增强、早停、更多数据、简化模型 |
| 正常 | 低 | 低 | — |

**通用调试顺序**：
1. 用 1 个 batch 验证能否过拟合到 100%（验证模型/代码正确）
2. 找合适 lr（range test）
3. 再调正则化
4. 最后调模型结构/规模

## 11.11 混合精度 AMP

In [ ]:
# AMP 用法（GPU 才能发挥效果，CPU 上不会报错）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = nn.Sequential(nn.Linear(100, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)
opt = torch.optim.Adam(model.parameters())
scaler = torch.amp.GradScaler(device.type if device.type == "cuda" else "cpu",
                              enabled=torch.cuda.is_available())

x = torch.randn(64, 100).to(device)
y = torch.randint(0, 10, (64,)).to(device)

opt.zero_grad()
with torch.autocast(device_type=device.type if device.type == "cuda" else "cpu",
                    enabled=torch.cuda.is_available()):
    logits = model(x)
    loss = nn.CrossEntropyLoss()(logits, y)
scaler.scale(loss).backward()
scaler.step(opt)
scaler.update()
print("AMP 训练一步完成，loss =", loss.item())

> `torch.cuda.amp` 已弃用，新写法是 `torch.amp.GradScaler` +
> `torch.autocast(device_type="cuda")`。

## 11.12 训练加速清单

| 手段 | 收益 |
|---|---|
| 混合精度 AMP | 省 30-50% 显存，提速 1.5-3x |
| 更大 batch + 调 lr | 充分利用 GPU |
| `channels_last` 内存格式 | CNN 提速 |
| `torch.compile(model)` | PyTorch 2.x 整体加速 1.3-2x |
| cudnn.benchmark=True | 固定输入尺寸时提速 |
| 预取数据 / 多 worker | 消除数据瓶颈 |
| 梯度检查点 | 用时间换显存 |

## 11.13 练习

**练习 1**：在 11.2 的梯度实验里加入残差连接，看梯度范数是否稳定。

**练习 2**：给 `TinyResNet` 训练 CIFAR 子集，比较有/无残差的收敛速度。

**练习 3**：给模型加 `Dropout(0.5)` + `weight_decay=1e-2`，观察过拟合是否缓解。

**练习 4**：实现 Label Smoothing 的损失，比较验证准确率。

**练习 5**：为实现一个简化版 Adam 优化器（自己写 update），并用它训练线性回归。

In [ ]:
# 练习 5：手写 Adam
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.params = list(params)
        self.lr = lr; self.b1, self.b2 = betas; self.eps = eps
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0
    def zero_grad(self):
        for p in self.params:
            p.grad = None
    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            g = p.grad
            self.m[i] = self.b1 * self.m[i] + (1 - self.b1) * g
            self.v[i] = self.b2 * self.v[i] + (1 - self.b2) * g ** 2
            m_hat = self.m[i] / (1 - self.b1 ** self.t)
            v_hat = self.v[i] / (1 - self.b2 ** self.t)
            with torch.no_grad():
                p -= self.lr * m_hat / (v_hat.sqrt() + self.eps)

w = torch.randn(1, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
opt = MyAdam([w, b], lr=0.1)
X = torch.rand(100, 1) * 4 - 2
Y = 3 * X + 1 + 0.1 * torch.randn(100, 1)
for _ in range(300):
    opt.zero_grad()
    loss = ((X @ w + b - Y) ** 2).mean()
    loss.backward(); opt.step()
print("练习5 手写 Adam 结果: w=%.3f b=%.3f (期望 3, 1)" % (w.item(), b.item()))

## 11.14 小结

- 反向传播 = 链式法则 + 复用中间梯度；PyTorch 自动完成。
- 梯度消失/爆炸来自 Jacobian 连乘；用**残差、合适初始化、归一化、梯度裁剪**解决。
- 初始化：ReLU 用 Kaiming，tanh 用 Xavier。
- 归一化：大 batch 用 BN，小 batch/序列用 LN/GN。
- 正则化：weight decay、dropout、数据增强、标签平滑、Mixup、EMA、早停。
- 优化器：默认 AdamW，CV 可 SGD+momentum；lr 是最关键超参数（warmup + cosine）。
- AMP + torch.compile 能显著加速。

下一节：`12_保存加载部署与性能优化.ipynb`